# Evaluation failure probe: six prefixes
Four actual looping states plus two controls from the interrupted evidence-v3 Planner evaluation. Exact saved messages; no ASR, tool execution, training, repair or enforcer. Step100 weights were not saved: use authentic step75 and retain historical step100 outputs for reference. This does NOT reproduce step100 exactly.
Test only the saved step75 adapter: base NF4 or FP16 crossed with explicit LoRA FP32 or FP16, using plain Transformers/PEFT with SDPA throughout. Batch1 and4, greedy max512 tokens,48 outputs total. Fresh process for each case on GPU0. Verify every LoRA parameter dtype before and after generation. No Unsloth or vLLM import. Startup/warmup costs are not controlled benchmarks.
Save source only; run yourself on T4 x2 with Internet. Outputs include exact token IDs, EOS/length stop, repetition indicators, prompt hashes, batch times and actual quantization/LoRA dtype settings. Reference answers are scored only after generation and never fed as generation input.


In [ ]:
from pathlib import Path
import os,sys,json,hashlib,subprocess,time
SOURCE=Path('/kaggle/working/eval_probe_source')
WORK=Path('/kaggle/working/evaluation_failure_probe_dtype2x2');WORK.mkdir(parents=True,exist_ok=True)
if not SOURCE.exists():subprocess.run(['git','clone','--filter=blob:none','--no-checkout','https://github.com/tnisl/baseline.git',str(SOURCE)],check=True)
subprocess.run(['git','-C',str(SOURCE),'fetch','origin','4894d331692293565635454a8c686b7c18314a04'],check=True)
subprocess.run(['git','-C',str(SOURCE),'checkout','--detach','FETCH_HEAD'],check=True)


In [ ]:
PAYLOAD={'requirements_train_2gpu.lock': 'accelerate==1.10.1\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   peft\n    #   trl\n    #   unsloth\n    #   unsloth-zoo\nagent-detector==2.0.0\n    # via fastapi-cloud-cli\naiohappyeyeballs==2.7.1\n    # via aiohttp\naiohttp==3.14.5\n    # via\n    #   fsspec\n    #   vllm\naiosignal==1.4.0\n    # via aiohttp\nannotated-doc==0.0.5\n    # via\n    #   fastapi\n    #   typer\nannotated-types==0.8.0\n    # via pydantic\nanyio==4.16.0\n    # via\n    #   httpx\n    #   httpx2\n    #   openai\n    #   starlette\n    #   watchfiles\nastor==0.8.1\n    # via depyf\nattrs==26.1.0\n    # via\n    #   aiohttp\n    #   jsonschema\n    #   referencing\nbitsandbytes==0.47.0\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   unsloth\nblake3==1.0.11\n    # via vllm\ncachetools==7.2.8\n    # via vllm\ncbor2==6.2.0.post2\n    # via vllm\ncertifi==2026.10.23\n    # via\n    #   httpcore\n    #   httpx\n    #   requests\n    #   sentry-sdk\ncffi==2.1.1\n    # via soundfile\ncharset-normalizer==3.5.2\n    # via requests\nclick==8.5.0\n    # via\n    #   ray\n    #   rich-toolkit\n    #   uvicorn\ncloudpickle==3.1.2\n    # via vllm\ncompressed-tensors==0.11.0\n    # via vllm\ncuda-pathfinder==1.8.5\n    # via cupy-cuda12x\ncupy-cuda12x==14.2.0\n    # via ray\ncut-cross-entropy==25.1.1\n    # via unsloth-zoo\ndatasets==3.6.0\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   trl\n    #   unsloth\n    #   unsloth-zoo\ndepyf==0.19.0\n    # via vllm\ndetect-installer==0.2.1\n    # via fastapi-cloud-cli\ndiffusers==0.35.1\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   unsloth\ndill==0.3.8\n    # via\n    #   datasets\n    #   depyf\n    #   multiprocess\ndiskcache==5.6.3\n    # via vllm\ndnspython==2.9.1\n    # via email-validator\ndocstring-parser==0.18.0\n    # via tyro\neinops==0.8.2\n    # via vllm\nemail-validator==2.3.0\n    # via\n    #   fastapi\n    #   pydantic\nfastapi==0.162.0\n    # via vllm\nfastapi-cli==0.0.32\n    # via fastapi\nfastapi-cloud-cli==0.26.0\n    # via fastapi-cli\nfastar==0.12.0\n    # via\n    #   fastapi\n    #   fastapi-cloud-cli\nfilelock==4.2.7\n    # via\n    #   datasets\n    #   diffusers\n    #   huggingface-hub\n    #   ray\n    #   torch\n    #   transformers\n    #   unsloth-zoo\n    #   vllm\nfrozendict==2.5.1\n    # via compressed-tensors\nfrozenlist==1.8.0\n    # via\n    #   aiohttp\n    #   aiosignal\nfsspec==2025.3.0\n    # via\n    #   datasets\n    #   huggingface-hub\n    #   torch\ngguf==0.19.0\n    # via vllm\ngoogleapis-common-protos==1.76.0\n    # via opentelemetry-exporter-otlp-proto-http\nh11==0.16.0\n    # via\n    #   httpcore\n    #   httpcore2\n    #   uvicorn\nhf-transfer==0.1.9\n    # via\n    #   unsloth\n    #   unsloth-zoo\nhf-xet==1.8.0\n    # via huggingface-hub\nhttpcore==1.0.9\n    # via httpx\nhttpcore2==2.16.0\n    # via httpx2\nhttptools==0.9.0\n    # via uvicorn\nhttpx==0.28.1\n    # via\n    #   fastapi\n    #   fastapi-cloud-cli\nhttpx2==2.16.0\n    # via\n    #   fastapi\n    #   openai\nhuggingface-hub==0.35.3\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   accelerate\n    #   datasets\n    #   diffusers\n    #   peft\n    #   tokenizers\n    #   transformers\n    #   unsloth\n    #   unsloth-zoo\nidna==3.20\n    # via\n    #   anyio\n    #   email-validator\n    #   httpx\n    #   httpx2\n    #   requests\n    #   yarl\nimportlib-metadata==10.1.0\n    # via diffusers\ninteregular==0.3.3\n    # via lm-format-enforcer\njinja2==3.1.6\n    # via\n    #   fastapi\n    #   torch\njiter==0.18.0\n    # via openai\njsonschema==4.26.0\n    # via\n    #   mistral-common\n    #   ray\njsonschema-specifications==2025.9.1\n    # via jsonschema\nlark==1.2.2\n    # via vllm\nllguidance==0.7.30\n    # via vllm\nllvmlite==0.44.0\n    # via numba\nlm-format-enforcer==0.11.3\n    # via vllm\nmarkdown-it-py==4.2.0\n    # via rich\nmarkupsafe==3.0.4\n    # via jinja2\nmdurl==0.1.2\n    # via markdown-it-py\nmistral-common==1.13.1\n    # via vllm\nmpmath==1.3.0\n    # via sympy\nmsgpack==1.2.4\n    # via ray\nmsgspec==0.22.0\n    # via\n    #   unsloth-zoo\n    #   vllm\nmultidict==7.2.1\n    # via\n    #   aiohttp\n    #   yarl\nmultiprocess==0.70.16\n    # via datasets\nnetworkx==3.7\n    # via torch\nninja==1.13.2\n    # via\n    #   vllm\n    #   xgrammar\nnumba==0.61.2\n    # via vllm\nnumpy==2.2.6\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   accelerate\n    #   bitsandbytes\n    #   cupy-cuda12x\n    #   datasets\n    #   diffusers\n    #   gguf\n    #   mistral-common\n    #   numba\n    #   opencv-python-headless\n    #   pandas\n    #   peft\n    #   scipy\n    #   soundfile\n    #   soxr\n    #   torchvision\n    #   transformers\n    #   unsloth\n    #   unsloth-zoo\n    #   vllm\n    #   xformers\n    #   xgrammar\nnvidia-cublas-cu12==12.8.4.1\n    # via\n    #   nvidia-cudnn-cu12\n    #   nvidia-cusolver-cu12\n    #   torch\nnvidia-cuda-cupti-cu12==12.8.90\n    # via torch\nnvidia-cuda-nvrtc-cu12==12.8.93\n    # via torch\nnvidia-cuda-runtime-cu12==12.8.90\n    # via torch\nnvidia-cudnn-cu12==9.10.2.21\n    # via torch\nnvidia-cufft-cu12==11.3.3.83\n    # via torch\nnvidia-cufile-cu12==1.13.1.3\n    # via torch\nnvidia-curand-cu12==10.3.9.90\n    # via torch\nnvidia-cusolver-cu12==11.7.3.90\n    # via torch\nnvidia-cusparse-cu12==12.5.8.93\n    # via\n    #   nvidia-cusolver-cu12\n    #   torch\nnvidia-cusparselt-cu12==0.7.1\n    # via torch\nnvidia-nccl-cu12==2.27.3\n    # via torch\nnvidia-nvjitlink-cu12==12.8.93\n    # via\n    #   nvidia-cufft-cu12\n    #   nvidia-cusolver-cu12\n    #   nvidia-cusparse-cu12\n    #   torch\nnvidia-nvtx-cu12==12.8.90\n    # via torch\nopenai==3.42.1\n    # via vllm\nopenai-harmony==0.0.8\n    # via vllm\nopencv-python-headless==5.0.0.93\n    # via\n    #   mistral-common\n    #   vllm\nopentelemetry-api==1.45.2\n    # via\n    #   fastapi\n    #   opentelemetry-exporter-http-transport\n    #   opentelemetry-exporter-otlp-proto-http\n    #   opentelemetry-sdk\n    #   opentelemetry-semantic-conventions\nopentelemetry-exporter-http-transport==0.66b2\n    # via opentelemetry-exporter-otlp-proto-http\nopentelemetry-exporter-otlp-common==0.66b2\n    # via opentelemetry-exporter-otlp-proto-http\nopentelemetry-exporter-otlp-proto-common==1.45.2\n    # via opentelemetry-exporter-otlp-proto-http\nopentelemetry-exporter-otlp-proto-http==1.45.2\n    # via fastapi\nopentelemetry-proto==1.45.2\n    # via\n    #   opentelemetry-exporter-otlp-proto-common\n    #   opentelemetry-exporter-otlp-proto-http\nopentelemetry-sdk==1.45.2\n    # via\n    #   fastapi\n    #   opentelemetry-exporter-otlp-common\n    #   opentelemetry-exporter-otlp-proto-http\nopentelemetry-semantic-conventions==0.66b2\n    # via opentelemetry-sdk\noutlines-core==0.2.11\n    # via vllm\npackaging==26.3\n    # via\n    #   accelerate\n    #   datasets\n    #   huggingface-hub\n    #   lm-format-enforcer\n    #   peft\n    #   ray\n    #   transformers\n    #   unsloth\n    #   unsloth-zoo\n    #   wheel\npandas==3.1.0\n    # via datasets\npartial-json-parser==0.2.1.1.post7\n    # via vllm\npeft==0.17.1\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   unsloth\n    #   unsloth-zoo\npillow==13.0.0\n    # via\n    #   diffusers\n    #   mistral-common\n    #   torchvision\n    #   unsloth-zoo\n    #   vllm\nprometheus-client==0.26.0\n    # via\n    #   prometheus-fastapi-instrumentator\n    #   vllm\nprometheus-fastapi-instrumentator==8.1.0\n    # via vllm\npropcache==0.5.4\n    # via\n    #   aiohttp\n    #   yarl\nprotobuf==7.36.2\n    # via\n    #   googleapis-common-protos\n    #   opentelemetry-proto\n    #   ray\n    #   unsloth\n    #   unsloth-zoo\n    #   vllm\npsutil==7.2.2\n    # via\n    #   accelerate\n    #   peft\n    #   unsloth\n    #   unsloth-zoo\n    #   vllm\npy-cpuinfo==9.0.0\n    # via vllm\npyarrow==26.0.0\n    # via datasets\npybase64==1.5.1\n    # via vllm\npycountry==26.2.16\n    # via pydantic-extra-types\npycparser==3.12\n    # via cffi\npydantic==2.14.2\n    # via\n    #   compressed-tensors\n    #   fastapi\n    #   fastapi-cloud-cli\n    #   lm-format-enforcer\n    #   mistral-common\n    #   openai\n    #   openai-harmony\n    #   pydantic-extra-types\n    #   pydantic-settings\n    #   vllm\n    #   xgrammar\npydantic-core==2.50.2\n    # via pydantic\npydantic-extra-types==2.11.1\n    # via\n    #   fastapi\n    #   mistral-common\npydantic-settings==2.16.2\n    # via fastapi\npygments==2.21.0\n    # via rich\npython-dateutil==4.0.0\n    # via pandas\npython-dotenv==1.2.4\n    # via\n    #   pydantic-settings\n    #   uvicorn\npython-json-logger==4.2.0\n    # via vllm\npython-multipart==0.0.33\n    # via fastapi\npyyaml==6.0.3\n    # via\n    #   accelerate\n    #   datasets\n    #   gguf\n    #   huggingface-hub\n    #   lm-format-enforcer\n    #   peft\n    #   ray\n    #   transformers\n    #   uvicorn\n    #   vllm\npyzmq==27.2.0\n    # via vllm\nray==2.59.0\n    # via vllm\nreferencing==0.37.0\n    # via\n    #   jsonschema\n    #   jsonschema-specifications\nregex==2026.9.29\n    # via\n    #   diffusers\n    #   tiktoken\n    #   transformers\n    #   unsloth-zoo\n    #   vllm\nrequests==2.34.2\n    # via\n    #   datasets\n    #   diffusers\n    #   gguf\n    #   huggingface-hub\n    #   mistral-common\n    #   opentelemetry-exporter-http-transport\n    #   opentelemetry-exporter-otlp-proto-http\n    #   ray\n    #   tiktoken\n    #   transformers\n    #   vllm\nrich==15.0.0\n    # via\n    #   rich-toolkit\n    #   typer\nrich-toolkit==0.20.6\n    # via\n    #   fastapi-cli\n    #   fastapi-cloud-cli\nrignore==0.8.1\n    # via fastapi-cloud-cli\nrpds-py==2026.9.1\n    # via\n    #   jsonschema\n    #   referencing\nsafetensors==0.8.0\n    # via\n    #   accelerate\n    #   diffusers\n    #   peft\n    #   transformers\nscipy==1.18.1\n    # via vllm\nsentencepiece==0.2.2\n    # via\n    #   unsloth\n    #   unsloth-zoo\n    #   vllm\nsentry-sdk==2.73.0\n    # via fastapi-cloud-cli\nsetproctitle==1.3.8\n    # via vllm\nsetuptools==79.0.1\n    # via\n    #   torch\n    #   triton\n    #   vllm\nshellingham==1.5.4\n    # via typer\nsix==1.17.0\n    # via vllm\nsniffio==1.3.1\n    # via openai\nsoundfile==0.14.0\n    # via mistral-common\nsoxr==1.1.0\n    # via mistral-common\nstarlette==1.10.0\n    # via\n    #   fastapi\n    #   prometheus-fastapi-instrumentator\nsympy==1.14.0\n    # via torch\ntiktoken==0.14.0\n    # via\n    #   mistral-common\n    #   vllm\ntokenizers==0.22.2\n    # via\n    #   transformers\n    #   vllm\ntorch==2.8.0\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   accelerate\n    #   bitsandbytes\n    #   compressed-tensors\n    #   cut-cross-entropy\n    #   peft\n    #   torchaudio\n    #   torchvision\n    #   unsloth\n    #   unsloth-zoo\n    #   vllm\n    #   xformers\n    #   xgrammar\ntorchao==0.13.0\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   unsloth-zoo\ntorchaudio==2.8.0\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   vllm\ntorchvision==0.23.0\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   unsloth\n    #   vllm\ntqdm==4.70.1\n    # via\n    #   datasets\n    #   gguf\n    #   huggingface-hub\n    #   peft\n    #   transformers\n    #   unsloth\n    #   unsloth-zoo\n    #   vllm\ntransformers==4.56.2\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   compressed-tensors\n    #   peft\n    #   trl\n    #   unsloth\n    #   unsloth-zoo\n    #   vllm\n    #   xgrammar\ntriton==3.4.0\n    # via\n    #   cut-cross-entropy\n    #   torch\n    #   unsloth\n    #   unsloth-zoo\n    #   xgrammar\ntrl==0.23.0\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   unsloth\n    #   unsloth-zoo\ntruststore==0.10.5\n    # via\n    #   httpcore2\n    #   httpx2\ntypeguard==4.6.0\n    # via tyro\ntyper==0.27.3\n    # via\n    #   fastapi-cli\n    #   fastapi-cloud-cli\ntyping-extensions==4.16.0\n    # via\n    #   aiohttp\n    #   aiosignal\n    #   anyio\n    #   fastapi\n    #   httpx2\n    #   huggingface-hub\n    #   openai\n    #   opentelemetry-api\n    #   opentelemetry-exporter-otlp-proto-http\n    #   opentelemetry-sdk\n    #   opentelemetry-semantic-conventions\n    #   pydantic\n    #   pydantic-core\n    #   pydantic-extra-types\n    #   pydantic-settings\n    #   referencing\n    #   rich-toolkit\n    #   soundfile\n    #   starlette\n    #   torch\n    #   typeguard\n    #   typing-inspection\n    #   tyro\n    #   unsloth-zoo\n    #   vllm\n    #   xgrammar\ntyping-inspection==0.4.4\n    # via\n    #   fastapi\n    #   pydantic\n    #   pydantic-settings\ntyro==1.0.16\n    # via\n    #   unsloth\n    #   unsloth-zoo\ntzdata==2026.6\n    # via python-dateutil\nunsloth==2025.10.9\n    # via -r /tmp/lifemate_stack_sources/requirements.in\nunsloth-zoo==2025.10.10\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   unsloth\nurllib3==2.8.0\n    # via\n    #   requests\n    #   sentry-sdk\nuvicorn==0.54.0\n    # via\n    #   fastapi\n    #   fastapi-cli\n    #   fastapi-cloud-cli\nuvloop==0.23.0\n    # via uvicorn\nvllm==0.10.2\n    # via -r /tmp/lifemate_stack_sources/requirements.in\nwatchfiles==1.3.0\n    # via\n    #   uvicorn\n    #   vllm\nwebsockets==17.4\n    # via uvicorn\nwheel==0.48.0\n    # via\n    #   unsloth\n    #   unsloth-zoo\nwrapt==1.17.3\n    # via -r /tmp/lifemate_stack_sources/requirements.in\nxformers==0.0.32.post1\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   unsloth\n    #   vllm\nxgrammar==0.1.23\n    # via vllm\nxxhash==4.0.2\n    # via datasets\nyarl==1.25.1\n    # via aiohttp\nzipp==4.1.2\n    # via importlib-metadata\n', 'umilife/eval_probe.py': '"""Small controlled generation probes; no optimizer, no ASR or tool execution."""\nimport argparse,json,os,time,hashlib,re\nfrom pathlib import Path\nfrom collections import Counter\nfrom umilife.core import read_jsonl,write_json\nfrom umilife.node_eval import assess\nMODEL=\'Qwen/Qwen3-4B-Instruct-2507\'\nREVISION=\'cdbee75f17c01a7cc42f958dc650907174af0554\'\nCASES={\'nf4_lora_fp32\':(\'nf4\',\'float32\'),\'nf4_lora_fp16\':(\'nf4\',\'float16\'),\n       \'fp16_lora_fp32\':(\'fp16\',\'float32\'),\'fp16_lora_fp16\':(\'fp16\',\'float16\')}\nBACKENDS=tuple(CASES)\nEOS_IDS=[151645,151643]\nHF_GENERATION={\'max_new_tokens\':512,\'do_sample\':False,\'num_beams\':1,\'use_cache\':True,\'logits_to_keep\':1,\n               \'pad_token_id\':151643,\'eos_token_id\':EOS_IDS,\'synced_gpus\':False,\'return_dict_in_generate\':False}\n\ndef model_settings(model):\n    quant=getattr(model.config,\'quantization_config\',None)\n    # Unsloth can expose a raw config dict with torch.dtype values instead of a BNB config object.\n    quant=json.loads(json.dumps(quant.to_dict() if hasattr(quant,\'to_dict\') else quant,default=str))\n    return {\'quantization_config\':quant,\n            \'lora_parameter_dtypes\':dict(Counter(str(p.dtype) for n,p in model.named_parameters() if \'lora_\' in n)),\n            \'attention_implementation\':getattr(model.config,\'_attn_implementation\',None)}\n\ndef fingerprint(ids):return hashlib.sha256(json.dumps(ids).encode()).hexdigest()\ndef repetition(raw):\n    words=re.findall(r\'\\w+\',raw.lower());counts=Counter(tuple(words[i:i+6]) for i in range(len(words)-5))\n    return max(counts.values(),default=0)\ndef trim_generated(ids,eos_ids):\n    for i,token in enumerate(ids):\n        if token in eos_ids:return ids[:i+1],\'eos\'\n    return ids,\'length\'\ndef groups(records,tokenizer,batch_size):\n    pairs=[(r,tokenizer.apply_chat_template(r[\'messages\'],tokenize=True,add_generation_prompt=True)) for r in records]\n    pairs.sort(key=lambda x:len(x[1]))\n    assert all(len(ids)+512<=8192 for _,ids in pairs)\n    return [pairs[i:i+batch_size] for i in range(0,len(pairs),batch_size)]\ndef save_rows(stream,group,tokenizer,outputs,catalog,backend,variant,batch_size,seconds,batch_index):\n    for (record,prompt_ids),(ids,finish,stop) in zip(group,outputs,strict=True):\n        raw=tokenizer.decode(ids,skip_special_tokens=True)\n        row={\'id\':record[\'id\'],\'step\':record[\'step\'],\'backend\':backend,\'variant\':variant,\'batch_size\':batch_size,\n             \'batch_index\':batch_index,\'batch_seconds\':seconds,\'input_tokens\':len(prompt_ids),\'input_ids_sha256\':fingerprint(prompt_ids),\n             \'generated_tokens\':len(ids),\'output_token_ids\':ids,\'finish_reason\':finish,\'stop_reason\':stop,\n             \'raw\':raw,\'max_repeated_6gram\':repetition(raw),**assess(\'planner\',record,raw,catalog)}\n        stream.write(json.dumps(row,ensure_ascii=False)+\'\\n\');stream.flush()\n        print(json.dumps({k:row[k] for k in (\'id\',\'step\',\'backend\',\'variant\',\'batch_size\',\'batch_index\',\'batch_seconds\',\'generated_tokens\',\'finish_reason\',\'schema_valid\',\'correct\',\'max_repeated_6gram\')}),flush=True)\n\ndef verify_lora_dtype(model,dtype):\n    params=[p for n,p in model.named_parameters() if \'lora_\' in n]\n    assert params and all(p.dtype==dtype for p in params),\'LoRA dtype changed or no adapter was loaded\'\n\ndef cast_lora_dtype(model,dtype):\n    # Cast only adapter parameters, never call model.to(dtype) on a quantized base.\n    for name,param in model.named_parameters():\n        if \'lora_\' in name:\n            param.data=param.data.to(dtype=dtype);param.requires_grad_(False)\n    verify_lora_dtype(model,dtype)\n\ndef restore_source_lora(model,weights,dtype):\n    from peft import set_peft_model_state_dict,get_peft_model_state_dict\n    import torch\n    result=set_peft_model_state_dict(model,weights)\n    assert not result.unexpected_keys and not [k for k in result.missing_keys if \'lora_\' in k]\n    actual=get_peft_model_state_dict(model)\n    assert set(actual)==set(weights),\'Adapter keys differ from the saved source\'\n    # Reload after casting: FP16 base loading may already have rounded adapter values.\n    for name,value in actual.items():\n        assert torch.equal(value.detach().cpu(),weights[name].to(dtype=dtype)),name\n    verify_lora_dtype(model,dtype)\n\ndef plain_hf(a,records,catalog,stream):\n    import sys,torch\n    from transformers import AutoModelForCausalLM,AutoTokenizer,BitsAndBytesConfig,set_seed\n    from peft import PeftModel\n    from peft.utils.save_and_load import load_peft_weights\n    assert not any(n==\'unsloth\' or n.startswith((\'unsloth.\',\'unsloth_zoo\')) for n in sys.modules),\'Plain Transformers must not import Unsloth patches\'\n    set_seed(42)\n    base_precision,lora_precision=CASES[a.backend]\n    config=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type=\'nf4\',bnb_4bit_use_double_quant=True,\n        bnb_4bit_compute_dtype=torch.float16) if base_precision==\'nf4\' else None\n    tokenizer=AutoTokenizer.from_pretrained(MODEL,revision=REVISION,trust_remote_code=False)\n    model=AutoModelForCausalLM.from_pretrained(MODEL,revision=REVISION,torch_dtype=torch.float16,\n        device_map={\'\':0},quantization_config=config,attn_implementation=\'sdpa\',trust_remote_code=False)\n    model=PeftModel.from_pretrained(model,str(Path(a.input)/\'adapter_step75\'),is_trainable=False,autocast_adapter_dtype=False)\n    before=model_settings(model)[\'lora_parameter_dtypes\']\n    requested=getattr(torch,lora_precision)\n    cast_lora_dtype(model,requested)\n    restore_source_lora(model,load_peft_weights(str(Path(a.input)/\'adapter_step75\'),device=\'cpu\'),requested)\n    model.eval();tokenizer.padding_side=\'left\';model.config.use_cache=True\n    assert not any(n==\'unsloth\' or n.startswith((\'unsloth.\',\'unsloth_zoo\')) for n in sys.modules)\n    write_json(Path(a.output)/f\'{a.backend}_{a.variant}_settings.json\',{\'effective_generation\':HF_GENERATION,\n        **model_settings(model),\'base_precision\':base_precision,\'requested_lora_dtype\':str(requested),\n        \'lora_dtypes_before_explicit_cast\':before,\'lora_source_weights_verified\':True,\'unsloth_imported\':False,\'adapter_step\':75})\n    for batch_size in (1,4):\n        for batch_index,group in enumerate(groups(records,tokenizer,batch_size)):\n            width=max(len(ids) for _,ids in group)\n            batch={\'input_ids\':torch.tensor([[151643]*(width-len(ids))+ids for _,ids in group],device=\'cuda\'),\n                   \'attention_mask\':torch.tensor([[0]*(width-len(ids))+[1]*len(ids) for _,ids in group],device=\'cuda\')}\n            torch.cuda.synchronize();started=time.monotonic()\n            with torch.inference_mode():out=model.generate(**batch,**HF_GENERATION)\n            torch.cuda.synchronize();seconds=time.monotonic()-started\n            verify_lora_dtype(model,requested)\n            outputs=[(*trim_generated(ids,set(EOS_IDS)),None) for ids in out[:,width:].cpu().tolist()]\n            save_rows(stream,group,tokenizer,outputs,catalog,a.backend,a.variant,batch_size,seconds,batch_index)\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--backend\',choices=BACKENDS,required=True);p.add_argument(\'--variant\',choices=(\'step75\',),required=True)\n    p.add_argument(\'--input\',required=True);p.add_argument(\'--output\',required=True);a=p.parse_args()\n    records=read_jsonl(Path(a.input)/\'records.jsonl\');catalog=json.loads((Path(a.input)/\'catalog.json\').read_text())\n    output=Path(a.output);output.mkdir(parents=True,exist_ok=True)\n    with (output/f\'{a.backend}_{a.variant}.jsonl\').open(\'w\') as stream:\n        plain_hf(a,records,catalog,stream)\nif __name__==\'__main__\':main()\n', 'umilife/node_eval.py': '"""Post-stage vLLM generation metrics for CE-best and last local checkpoints."""\nimport argparse,json,os,sys,subprocess,time,re,shutil\nfrom pathlib import Path\nfrom collections import Counter\nfrom jsonschema import Draft202012Validator,ValidationError\nfrom umilife.core import read_jsonl,write_json,parse_json,load_catalog,digest as file_digest\nfrom umilife.prompt_policy import request_schema,digest\nfrom umilife.evaluate import matches\n\ndef assess(role,record,raw,catalog):\n    target=parse_json(record[\'target\']);messages=json.loads(record[\'messages\'][1][\'content\'])\n    if role==\'planner\':\n        names=re.findall(r\'^\\s*"([^"\\n]+)"\\(args:\',messages[\'tools_ts\'],re.M)\n        specs={name:catalog[name] for name in names}\n        schema=request_schema(\'planner\',specs)\n    else:schema=request_schema(\'caller\',{target[\'name\']:catalog[target[\'name\']]},{\'tool\':target[\'name\']})\n    value=None;valid=False;error=None\n    try:\n        value=parse_json(raw);Draft202012Validator(schema).validate(value)\n        if role==\'planner\' and value.get(\'next\')==\'caller\' and (not value.get(\'tool\') or value[\'tool\'] not in catalog):raise ValueError(\'Missing/unknown selected tool\')\n        valid=True\n    except (ValueError,ValidationError) as exc:error=str(exc)[:300]\n    result={\'schema_valid\':valid,\'schema_error\':error,\'parsed\':value,\'target\':target}\n    if role==\'planner\':\n        route=bool(valid and value[\'next\']==target[\'next\']);tool=bool(valid and value.get(\'tool\')==target.get(\'tool\')) if target[\'next\']==\'caller\' else None\n        result.update(correct=bool(route and (target[\'next\']!=\'caller\' or tool)),route_match=route,tool_match=tool,terminal=target[\'next\']!=\'caller\')\n    else:\n        correct=bool(valid and matches(value,target,catalog));fields={}\n        pred=value.get(\'arguments\',{}) if valid else {}\n        for k,v in target[\'arguments\'].items():\n            fields[k]=bool(valid and k in pred and matches({\'name\':value[\'name\'],\'arguments\':{k:pred[k]}},{\'name\':target[\'name\'],\'arguments\':{k:v}},catalog))\n        count=sum(fields.values());denom=len(pred)+len(target[\'arguments\'])\n        result.update(correct=correct,field_matches=fields,argument_f1=2*count/denom if denom else float(valid and value[\'name\']==target[\'name\']))\n    return result\n\ndef summarize(rows,role):\n    n=len(rows);assert n\n    report={\'samples\':n,\'match_accuracy\':sum(r[\'correct\'] for r in rows)/n,\'schema_valid_rate\':sum(r[\'schema_valid\'] for r in rows)/n}\n    if role==\'planner\':\n        report[\'route_accuracy\']=sum(r[\'route_match\'] for r in rows)/n\n        calls=[r for r in rows if not r[\'terminal\']]\n        report[\'tool_accuracy\']=sum(r[\'tool_match\'] for r in calls)/len(calls) if calls else None\n        report[\'tool_samples\']=len(calls)\n        for kind in (\'initial\',\'intermediate\',\'terminal\'):\n            sub=[r for r in rows if (\'terminal\' if r[\'terminal\'] else \'initial\' if r[\'step\']==0 else \'intermediate\')==kind]\n            report[kind]={\'n\':len(sub),\'accuracy\':sum(r[\'correct\'] for r in sub)/len(sub) if sub else None}\n        report[\'continued_after_reference_end\']=sum(r[\'terminal\'] and isinstance(r[\'parsed\'],dict) and r[\'parsed\'].get(\'next\')==\'caller\' for r in rows)\n    else:\n        fields={}\n        for row in rows:\n            for k,v in row[\'field_matches\'].items():fields.setdefault(k,{\'n\':0,\'correct\':0});fields[k][\'n\']+=1;fields[k][\'correct\']+=v\n        for value in fields.values():value[\'accuracy\']=value[\'correct\']/value[\'n\']\n        report.update(argument_f1=sum(r[\'argument_f1\'] for r in rows)/n,fields=fields)\n    return report\n\ndef worker(a):\n    from umilife.t4_compat import prepare_t4_lora_kernel\n    print(json.dumps(prepare_t4_lora_kernel()),flush=True)\n    import torch\n    from vllm import LLM,SamplingParams\n    from vllm.lora.request import LoRARequest\n    from umilife.runtime_options import engine_options\n    work=Path(a.output);config=json.loads((work/\'config.json\').read_text());options=engine_options(torch.cuda.get_device_capability(0))\n    engine=LLM(model=a.model,revision=a.revision,tokenizer=config[\'tokenizer\'],dtype=\'float16\',tensor_parallel_size=1,\n        enable_lora=True,max_loras=2,max_cpu_loras=2,max_lora_rank=16,max_model_len=8192,max_num_seqs=4,max_num_batched_tokens=8192,\n        gpu_memory_utilization=.85,enforce_eager=not options[\'cuda_graphs\'],enable_prefix_caching=options[\'prefix_caching\'],\n        enable_chunked_prefill=False,seed=42,generation_config=\'vllm\',disable_log_stats=True)\n    tokenizer=engine.get_tokenizer();records=read_jsonl(Path(a.prepared)/f\'{a.stage}.validation.jsonl\')[a.rank::2]\n    catalog=load_catalog(a.data_root,\'train\');sampling=SamplingParams(temperature=0,seed=42,max_tokens=512,guided_decoding=None)\n    candidates={name:LoRARequest(name,i+1,path) for i,(name,path) in enumerate(config[\'candidates\'].items())}\n    target=work/f\'gpu{a.rank}.jsonl\'\n    with target.open(\'w\') as out:\n        for start in range(0,len(records),4):\n            group=records[start:start+4]\n            prompts=[tokenizer.apply_chat_template(r[\'messages\'],tokenize=False,add_generation_prompt=True) for r in group]\n            token_ids=[tokenizer.encode(p,add_special_tokens=False) for p in prompts]\n            assert all(len(ids)+512<=8192 for ids in token_ids),\'Context exceeded; no evidence truncation\'\n            order=list(candidates)\n            if start//4%2:order.reverse()\n            for name in order:\n                tick=time.monotonic();results=engine.generate(prompts,sampling,lora_request=candidates[name],use_tqdm=False)\n                for record,ids,result in zip(group,token_ids,results,strict=True):\n                    row={\'id\':record[\'id\'],\'step\':record[\'step\'],\'candidate\':name,\'prompt_sha256\':digest(record[\'messages\']),\n                        \'input_ids_sha256\':digest(ids),\'raw\':result.outputs[0].text,\'finish_reason\':result.outputs[0].finish_reason,\n                        **assess(a.stage,record,result.outputs[0].text,catalog)}\n                    out.write(json.dumps(row,ensure_ascii=False)+\'\\n\');out.flush()\n                print(json.dumps({\'stage\':a.stage,\'rank\':a.rank,\'candidate\':name,\'completed\':min(start+4,len(records)),\n                                  \'total\':len(records),\'batch_seconds\':time.monotonic()-tick}),flush=True)\n\ndef export_adapter(adapter,weights,export,lineage,report):\n    # Export a self-contained inference adapter without overwriting the CE-best training artifact.\n    temp=export.with_name(export.name+\'.tmp\')\n    if temp.exists():shutil.rmtree(temp)\n    temp.mkdir(parents=True)\n    for path in adapter.iterdir():\n        if path.is_file() and path.name not in (\'adapter_model.safetensors\',\'lineage.json\'):shutil.copy2(path,temp/path.name)\n    shutil.copy2(weights,temp/weights.name)\n    import torch\n    from safetensors import safe_open\n    parameters=0\n    with safe_open(weights,framework=\'pt\',device=\'cpu\') as handle:\n        for key in handle.keys():\n            value=handle.get_tensor(key);assert torch.isfinite(value).all(),key;parameters+=value.numel()\n    selected_lineage={**lineage,\'selection\':report,\n        \'ce_best_final_adapter\':lineage[\'final_adapter\'],\n        \'final_adapter\':{\'sha256\':file_digest(weights),\'hash_kind\':\'safetensors_file\',\'finite\':True,\'parameters\':parameters}}\n    write_json(temp/\'lineage.json\',selected_lineage)\n    if export.exists():shutil.rmtree(export)\n    temp.rename(export)\n    return export\n\ndef main():\n    p=argparse.ArgumentParser()\n    for name in (\'stage\',\'prepared\',\'adapter\',\'checkpoints\',\'data-root\',\'model\',\'revision\',\'output\'):p.add_argument(\'--\'+name,required=True)\n    p.add_argument(\'--rank\',type=int)\n    a=p.parse_args()\n    if a.rank is not None:worker(a);return\n    work=Path(a.output);work.mkdir(parents=True,exist_ok=True);adapter=Path(a.adapter);cp=Path(a.checkpoints)\n    latest=cp/json.loads((cp/\'latest.json\').read_text())[\'folder\']\n    lineage=json.loads((adapter/\'lineage.json\').read_text())\n    assert lineage[\'stage\']==a.stage and lineage[\'model\']==a.model and lineage[\'revision\']==a.revision\n    assert lineage[\'dataset_manifest_sha256\']==file_digest(Path(a.prepared)/\'manifest.json\')\n    candidates={\'ce_best\':str(adapter),\'last\':str(latest)}\n    if file_digest(adapter/\'adapter_model.safetensors\')==file_digest(latest/\'adapter_model.safetensors\'):candidates.pop(\'last\')\n    config={\'stage\':a.stage,\'model\':a.model,\'revision\':a.revision,\'tokenizer\':str(adapter),\'candidates\':candidates,\n        \'weights_sha256\':{k:file_digest(Path(v)/\'adapter_model.safetensors\') for k,v in candidates.items()},\n        \'manifest_sha256\':file_digest(Path(a.prepared)/\'manifest.json\'),\'prompt_sha256\':json.loads((Path(a.prepared)/\'manifest.json\').read_text())[\'prompts_sha256\']}\n    if (work/\'report.json\').exists():\n        assert json.loads((work/\'config.json\').read_text())==config,\'Stale node eval output\'\n        print((work/\'report.json\').read_text());return\n    write_json(work/\'config.json\',config)\n    processes=[];logs=[];last_seen=[None,None]\n    try:\n        for rank in range(2):\n            env=os.environ.copy();env.update(CUDA_VISIBLE_DEVICES=str(rank),TRITON_CACHE_DIR=f\'/tmp/node_eval_triton_{rank}\')\n            log=(work/f\'gpu{rank}.log\').open(\'w\');logs.append(log)\n            command=[sys.executable,\'-u\',\'-m\',\'umilife.node_eval\',*sys.argv[1:],\'--rank\',str(rank)]\n            processes.append(subprocess.Popen(command,env=env,stdout=log,stderr=subprocess.STDOUT))\n        while any(p.poll() is None for p in processes):\n            if any(p.poll() not in (None,0) for p in processes):raise RuntimeError(\'Node generation worker failed\')\n            for rank,log in enumerate(logs):\n                log.flush()\n                for line in reversed((work/f\'gpu{rank}.log\').read_text().splitlines()[-12:]):\n                    try:event=json.loads(line)\n                    except ValueError:continue\n                    if isinstance(event,dict) and \'completed\' in event:\n                        if line!=last_seen[rank]:print(json.dumps({\'node_eval_progress\':event}),flush=True);last_seen[rank]=line\n                        break\n            time.sleep(5)\n        assert all(p.returncode==0 for p in processes),\'Node generation worker failed\'\n    except BaseException:\n        for process in processes:\n            if process.poll() is None:process.terminate()\n        for rank in range(len(logs)):\n            logs[rank].flush();print((work/f\'gpu{rank}.log\').read_text()[-6000:],flush=True)\n        raise\n    finally:\n        for process in processes:\n            if process.poll() is None:\n                try:process.wait(timeout=20)\n                except subprocess.TimeoutExpired:process.kill();process.wait()\n        for log in logs:log.close()\n    rows=[r for rank in range(2) for r in read_jsonl(work/f\'gpu{rank}.jsonl\')]\n    expected={(r[\'id\'],r[\'step\']) for r in read_jsonl(Path(a.prepared)/f\'{a.stage}.validation.jsonl\')}\n    for name in candidates:\n        selected=[r for r in rows if r[\'candidate\']==name]\n        assert len(selected)==len(expected) and {(r[\'id\'],r[\'step\']) for r in selected}==expected\n    for key in expected:\n        pair=[r for r in rows if (r[\'id\'],r[\'step\'])==key]\n        assert len({(r[\'prompt_sha256\'],r[\'input_ids_sha256\']) for r in pair})==1\n    metrics={name:summarize([r for r in rows if r[\'candidate\']==name],a.stage) for name in candidates}\n    winner=max(metrics,key=lambda k:(metrics[k][\'match_accuracy\'],metrics[k][\'schema_valid_rate\'],k==\'ce_best\'))\n    report={\'stage\':a.stage,\'metrics\':metrics,\'recommended_checkpoint\':winner,\'recommended_path\':candidates[winner],\n            \'guided_decoding\':False,\'batch_per_gpu\':4,\'gpu_workers\':2,\n            \'note\':\'Reference-prefix node agreement, not full workflow/contest score. CE-best and last only; no claim of globally best checkpoint.\'}\n    export=work.parent.parent/\'selected_adapters\'/a.stage\n    export_adapter(adapter,Path(candidates[winner])/\'adapter_model.safetensors\',export,lineage,report)\n    report[\'exported_adapter\']=str(export)\n    write_json(work/\'report.json\',report);print(json.dumps(report,ensure_ascii=False,indent=2),flush=True)\n\nif __name__==\'__main__\':main()\n', 'umilife/probe_summary.py': 'import argparse,json\nfrom pathlib import Path\nfrom umilife.eval_probe import BACKENDS,CASES\nfrom umilife.core import read_jsonl,write_json\n\ndef summarize_probe(root,source):\n    rows=[r for backend in BACKENDS for variant in (\'step75\',) for r in read_jsonl(root/f\'{backend}_{variant}.jsonl\')]\n    selected=json.loads((source/\'probe_manifest.json\').read_text());keys={(r[\'id\'],r[\'step\']) for r in selected[\'samples\']}\n    report={\'adapter_step\':75,\'historical_step\':100,\'exact_step100_reproduction\':False,\'cases\':{},\'paired\':[],\n            \'limitations\':[\'Six failure-selected/control prefixes, not a validation score.\',\'Plain Transformers/SDPA only; explicit LoRA cast verified before and after generation. Kernel and rounding effects remain possible.\',\'Warmup/compilation/cache and case order affect batch timings.\',\'Step75 weights cannot reproduce the lost step100 weights exactly.\']}\n    for backend in BACKENDS:\n        for variant in (\'step75\',):\n            for batch in (1,4):\n                group=[r for r in rows if (r[\'backend\'],r[\'variant\'],r[\'batch_size\'])==(backend,variant,batch)]\n                assert len(group)==len(keys) and {(r[\'id\'],r[\'step\']) for r in group}==keys\n                times={r[\'batch_index\']:r[\'batch_seconds\'] for r in group}\n                report[\'cases\'][f\'{backend}/{variant}/batch{batch}\']={\'samples\':len(group),\'schema_valid\':sum(r[\'schema_valid\'] for r in group),\n                    \'matches\':sum(r[\'correct\'] for r in group),\'repeated_6gram_ge4\':sum(r[\'max_repeated_6gram\']>=4 for r in group),\n                    \'length_limited\':sum(r[\'finish_reason\']==\'length\' for r in group),\'generated_tokens\':sum(r[\'generated_tokens\'] for r in group),\n                    \'generation_seconds\':sum(times.values())}\n    for key in sorted(keys):\n        group=[r for r in rows if (r[\'id\'],r[\'step\'])==key]\n        assert len({r[\'input_ids_sha256\'] for r in group})==1,\'Prompt/tokenizer parity failed\'\n        by={(r[\'backend\'],r[\'variant\'],r[\'batch_size\']):r for r in group}\n        report[\'paired\'].append({\'id\':key[0],\'step\':key[1],\'prompt_token_ids_identical\':True,\n            \'batch1_vs4_identical\':{f\'{b}/{v}\':by[b,v,1][\'raw\']==by[b,v,4][\'raw\'] for b in BACKENDS for v in (\'step75\',)},\n            \'same_base_different_lora_dtype\':{f\'{base}/batch{n}\':by[f\'{base}_lora_fp32\',\'step75\',n][\'raw\']==by[f\'{base}_lora_fp16\',\'step75\',n][\'raw\'] for base in (\'nf4\',\'fp16\') for n in (1,4)},\n            \'same_lora_dtype_different_base\':{f\'{dtype}/batch{n}\':by[f\'nf4_lora_{dtype}\',\'step75\',n][\'raw\']==by[f\'fp16_lora_{dtype}\',\'step75\',n][\'raw\'] for dtype in (\'fp32\',\'fp16\') for n in (1,4)}})\n    for backend in BACKENDS:\n        settings=json.loads((root/f\'{backend}_step75_settings.json\').read_text())\n        wanted=\'torch.\'+CASES[backend][1]\n        assert settings[\'requested_lora_dtype\']==wanted\n        assert set(settings[\'lora_parameter_dtypes\'])=={wanted},\'Actual LoRA dtype mismatch\'\n        assert settings[\'lora_source_weights_verified\'] is True\n        assert settings[\'attention_implementation\']==\'sdpa\' and settings[\'unsloth_imported\'] is False\n        report.setdefault(\'verified_settings\',{})[backend]=settings\n    write_json(root/\'report.json\',report)\n    text=[\'# Evaluation failure probe\',\'\', \'Step75 adapter; historical failures were at step100. No training or enforcer. 2x2 base NF4/FP16 x LoRA FP32/FP16, plain Transformers/SDPA.\',\'\',\n        \'| Case | Valid JSON/schema | Match | Repetition | Length limit | Generation seconds |\',\'|---|---:|---:|---:|---:|---:|\']\n    for name,r in report[\'cases\'].items():text.append(f"| {name} | {r[\'schema_valid\']}/6 | {r[\'matches\']}/6 | {r[\'repeated_6gram_ge4\']} | {r[\'length_limited\']} | {r[\'generation_seconds\']:.2f} |")\n    text+=[\'\',\'Read report.json for paired comparisons; raw JSONL files include exact output token IDs and finish/stop reasons. Historical outputs are in the input dataset. Actual dtype/quantization settings and dtype verification failures are in per-case JSON/log files.\',\'\',*report[\'limitations\']]\n    (root/\'report.md\').write_text(\'\\n\'.join(text)+\'\\n\');print(\'\\n\'.join(text),flush=True)\n    return report\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--work\',required=True);p.add_argument(\'--input\',required=True);a=p.parse_args();summarize_probe(Path(a.work),Path(a.input))\n', 'umilife/prompt_policy.py': '"""Generic tool policy. No benchmark examples, held-out labels or per-sample rules."""\nfrom .typescript_schema import typescript_catalog\nfrom copy import deepcopy\nfrom datetime import datetime,timedelta\nimport hashlib,json\n\nPOLICY_VERSION=\'planner-caller-evidence-v3\'\nCOMMON="""You are LifeMate, a personal assistant using installed tools.\nAct only on the current request. History resolves references and may satisfy explicit already-completed or recall conditions, but is not a newly executed operation in this turn. Preserve quantities, negation, conditions and exact text to send or save. Use the supplied clock and timezone. Never invent identifiers or tool results.\nOnly current observations confirm execution. Do not repeat successful effects, even with reworded arguments. Retry only with evidence of failure and a justified correction. Reuse unchanged read results; change strategy after an unhelpful lookup.\nTool contracts are TypeScript declarations: ? marks optional fields, literal unions specify enums, and comments describe constraints and identifier sources. These are interface documents, not executable code. Return one JSON object without markdown.\n"""\nPROMPTS={\n \'planner\':COMMON+"""You are Planner. Decide the next single step toward pending user objectives.\nChoose the installed tool, order dependent operations, evaluate conditions from available evidence, and stop when no further authorized operation is needed. If a condition or required identifier needs discovery, select an appropriate read tool first. Choose sys.ask_missing_info when required information cannot be resolved from context or supported lookups. An unresolved ID alone is not an unsupported request.\nWrite thought in English, normally one to three sentences. Identify the pending objective, decisive user constraints and the visible evidence supporting the next step. Explain why the selected operation is appropriate, not just its name or parameter sources. Keep selection predicates, negation and the latest correction. You may state observed identifiers or derived quantities/times when justified; do not invent evidence or claim an unexecuted result. Resolve business conditions and choose the intended entity; Caller binds that decision to the selected contract and verifies values against the original request. Distinguish modifying an existing object from creating another one. Track which objectives were completed and what remains. End only with supporting observations or an explicit no-op/unsupported reason. Reuse a successful unchanged lookup after repeated-read feedback; repeating its name is not progress.\nFor an operation return {thought,next:"caller",tool}. For a completed/no-op turn or after requesting clarification return {thought,next:"summarizer"}; this terminal route does not require a trained Summarizer. For an unsupported request return {thought,next:"give_up"}. Never stop while a supported, authorized objective remains, repeat a confirmed write for a reply, or claim unconfirmed success.\n""",\n \'caller\':COMMON+"""You are Caller. Convert Planner\'s selected step into one valid API call.\nReturn exactly {name,arguments}; name must equal plan.tool. Planner owns routing and tool selection. Resolve values using the current request, history, context and previous tool observations. Treat plan.thought as guidance, not proof. Match the intended entity before taking its ID; use identifiers actually observed or explicitly supplied. Respect user qualifiers rather than choosing the first result. Normalize dates with the supplied clock/timezone and preserve exact text to send or save.\nRead the selected TypeScript contract, comments, required/optional fields, enums and alternative argument groups. Supply only supported values. Optional in the API does not mean optional for the user intent: include every field needed to express the requested schedule, end time, filter, type or state. Omit optional fields only when they have no supporting requirement. A latest correction overrides the earlier value. Distinguish off from changing speed/brightness. Convert relative dates and clock expressions using the supplied clock/timezone; verify arithmetic rather than copying an unrelated time. Preserve the requested message facts and do not invent recipients or numbers. For sys.ask_missing_info, use the intended tool_name and exact missing parameter names from its contract. Never invent a required identifier or change the selected tool. Correct validation feedback without repeating confirmed effects. Output no thought, next or answer. The runtime executes your JSON.\n""",\n \'summarizer\':COMMON+"Return only {answer}, briefly grounded in actual observations. Do not claim unconfirmed success.",\n \'global\':COMMON+"Handle one ReAct step: return {thought,next,tool,action} for caller, {thought,next,answer} for summarizer, or {thought,next} for give_up. Keep the English thought concise; action contains {name,arguments}."\n}\nCOMPLETION_REVIEW="""Review this proposed ending against every objective in the current request. Has something only been looked up but not executed? Are you confusing history with current execution? Are there unfinished objectives, actual operation failures, or unresolved empty searches? Do not mistake business-state fields for execution failures. Do not propose repeating any confirmed effect just to verify it or generate a reply; summarize when all objectives are satisfied. A different argument spelling does not justify repeating a completed objective. Continue with caller when work remains. Prefer lookup before clarification. Summarize only with sufficient evidence or while waiting for clarification; give_up only beyond installed capabilities."""\n\ndef canonical(value):return json.dumps(value,ensure_ascii=False,sort_keys=True,separators=(\',\',\':\'))\ndef digest(value):return hashlib.sha256(canonical(value).encode()).hexdigest()\ndef obj(properties,required):return {\'type\':\'object\',\'properties\':properties,\'required\':required,\'additionalProperties\':False}\n\ndef parameter_schema(schema):\n    """Convert documented x-one-of to ordinary JSON Schema; preserve typed nesting."""\n    allowed={\'type\',\'properties\',\'required\',\'items\',\'enum\',\'format\',\'minimum\',\'maximum\',\n             \'exclusiveMinimum\',\'exclusiveMaximum\',\'minLength\',\'maxLength\',\'minItems\',\'maxItems\',\n             \'anyOf\',\'oneOf\',\'allOf\',\'additionalProperties\',\'description\'}\n    out={}\n    for key,value in schema.items():\n        if key not in allowed:continue\n        if key==\'properties\':out[key]={name:parameter_schema(prop) for name,prop in value.items()}\n        elif key==\'items\' and isinstance(value,dict):out[key]=parameter_schema(value)\n        elif key in (\'anyOf\',\'oneOf\',\'allOf\'):out[key]=[parameter_schema(x) for x in value]\n        else:out[key]=deepcopy(value)\n    if out.get(\'type\')==\'object\' and \'properties\' in out:out[\'additionalProperties\']=False\n    if schema.get(\'x-one-of\'):\n        # Full object alternatives are supported by both JSON Schema and LMFE.\n        base=out\n        out={\'anyOf\':[{**deepcopy(base),\'required\':list(dict.fromkeys(base.get(\'required\',[])+list(group)))}\n                      for group in schema[\'x-one-of\']]}\n    return out\n\ndef action_schema(specs,name):\n    arguments=parameter_schema(specs[name][\'parameters\'])\n    if name==\'sys.ask_missing_info\':\n        # Only public installed schemas provide these choices, never an answer label.\n        choices=[n for n,s in specs.items() if n!=name and s[\'parameters\'].get(\'properties\')]\n        if choices:arguments[\'properties\'][\'tool_name\'][\'enum\']=sorted(choices)\n        arguments[\'properties\'][\'params\'][\'minItems\']=1\n        fields=sorted({key for target in choices for key in specs[target][\'parameters\'][\'properties\']})\n        if fields:arguments[\'properties\'][\'params\'][\'items\'][\'enum\']=fields\n    return obj({\'name\':{\'type\':\'string\',\'enum\':[name]},\'arguments\':arguments},[\'name\',\'arguments\'])\n\ndef route_schema(specs,global_mode=False):\n    thought={\'type\':\'string\',\'maxLength\':1024};branches=[]\n    if specs:\n        if global_mode:\n            for name in sorted(specs):\n                branches.append(obj({\'thought\':thought,\'next\':{\'type\':\'string\',\'enum\':[\'caller\']},\n                    \'tool\':{\'type\':\'string\',\'enum\':[name]},\'action\':action_schema(specs,name)},[\'thought\',\'next\',\'tool\',\'action\']))\n        else:\n            branches.append(obj({\'thought\':thought,\'next\':{\'type\':\'string\',\'enum\':[\'caller\']},\n                \'tool\':{\'type\':\'string\',\'enum\':sorted(specs)}},[\'thought\',\'next\',\'tool\']))\n    done={\'thought\':thought,\'next\':{\'type\':\'string\',\'enum\':[\'summarizer\']}}\n    if global_mode:done[\'answer\']={\'type\':\'string\',\'maxLength\':1024}\n    branches.append(obj(done,list(done)))\n    branches.append(obj({\'thought\':thought,\'next\':{\'type\':\'string\',\'enum\':[\'give_up\']}},[\'thought\',\'next\']))\n    return {\'anyOf\':branches}\n\ndef compact_catalog(specs):\n    fields=(\'type\',\'enum\',\'x-ref\',\'format\',\'items\',\'minimum\',\'maximum\')\n    return [{\'name\':name,\'kind\':spec.get(\'kind\'),\'description\':spec[\'description\'],\n             \'parameters\':{key:{k:deepcopy(v) for k,v in prop.items() if k in fields}\n                           for key,prop in spec[\'parameters\'].get(\'properties\',{}).items()},\n             \'required\':spec[\'parameters\'].get(\'required\',[]),\n             **({\'one_of\':spec[\'parameters\'][\'x-one-of\']} if spec[\'parameters\'].get(\'x-one-of\') else {})}\n            for name,spec in sorted(specs.items())]\n\ndef reference_clock(context):\n    result={}\n    try:\n        now=datetime.fromisoformat(context[\'current_time\'])\n        result={\'today\':now.date().isoformat(),\'tomorrow\':(now+timedelta(days=1)).date().isoformat(),\n                \'day_after_tomorrow\':(now+timedelta(days=2)).date().isoformat(),\n                \'weekday_iso\':now.isoweekday(),\'timezone\':context.get(\'timezone\')}\n    except (KeyError,TypeError,ValueError):pass\n    return result\n\ndef build_messages(role,row,text,specs,trajectory,plan=None,feedback=None):\n    # Explicit provenance allowlist. Current labels/final state/reference transcript never enter here.\n    context={k:deepcopy(row[\'context\'][k]) for k in (\'current_time\',\'timezone\',\'location\',\'user_profile\') if k in row[\'context\']}\n    history=[{k:deepcopy(turn[k]) for k in (\'role\',\'text\',\'tool_calls\',\'tool_results\') if k in turn}\n             for turn in row.get(\'history\',[])]\n    content={\'context\':context,\'reference_clock\':reference_clock(context),\'history\':history,\n             \'request\':text,\'trajectory\':deepcopy(trajectory)}\n    if role in (\'planner\',\'global\'):content[\'tools_ts\']=typescript_catalog(specs)\n    elif role==\'caller\':\n        selected=(plan or {}).get(\'tool\');assert selected in specs,\'Planner selected an unavailable tool\'\n        content.update(plan=deepcopy(plan),selected_tool_ts=typescript_catalog({selected:specs[selected]},detailed=True))\n        if selected==\'sys.ask_missing_info\':\n            content[\'missing_information_tools_ts\']=typescript_catalog({name:spec for name,spec in specs.items()\n                if name!=selected and spec[\'parameters\'].get(\'properties\')})\n    else:content[\'plan\']=deepcopy(plan)\n    if feedback is not None:content[\'validation_feedback\']=deepcopy(feedback)\n    # Keep the original request at the end, after schemas/observations/feedback.\n    content.pop(\'request\');content[\'request\']=text\n    return [{\'role\':\'system\',\'content\':PROMPTS[role]},\n            {\'role\':\'user\',\'content\':json.dumps(content,ensure_ascii=False,separators=(\',\',\':\'))}]\n\ndef request_schema(role,specs,plan=None):\n    if role==\'caller\':return action_schema(specs,plan[\'tool\'])\n    if role in (\'planner\',\'global\'):return route_schema(specs,role==\'global\')\n    return obj({\'answer\':{\'type\':\'string\',\'maxLength\':1024}},[\'answer\'])\n\ndef validate_meta_action(action,specs):\n    if action[\'name\']!=\'sys.ask_missing_info\':return None\n    args=action[\'arguments\'];target=args.get(\'tool_name\');params=args.get(\'params\',[])\n    if target not in specs or target==\'sys.ask_missing_info\':return \'Clarification target must be another installed tool\'\n    # An optional API parameter can still be necessary to express the user\'s intent.\n    fields=set(specs[target][\'parameters\'].get(\'properties\',{}))\n    if not params or len(params)!=len(set(params)) or not set(params)<=fields:\n        return \'params must contain unique parameter names from the target tool contract\'\n    return None\n\ndef repeated_read(action,specs,trajectory):\n    if specs[action[\'name\']].get(\'kind\')!=\'read\':return False\n    for old in reversed(trajectory):\n        if specs.get(old[\'action\'][\'name\'],{}).get(\'kind\')==\'write\':return False\n        if canonical(old[\'action\'])==canonical(action):return True\n    return False\n\nPOLICY_AUDIT={\'policy_version\':POLICY_VERSION,\'static_prompts_sha256\':digest(PROMPTS),\n    \'completion_review_sha256\':digest(COMPLETION_REVIEW),\'few_shot_examples\':0,\n    \'sample_specific_rules\':False,\'gold_inputs_to_model\':False,\n    \'allowed_input_sources\':[\'current ASR request\',\'ASR user history\',\'provided context\',\n        \'installed public tool schemas\',\'observations from executed tools\',\'model-generated plan/feedback\']}\n', 'umilife/typescript_schema.py': '"""Compact TypeScript interfaces derived solely from the installed public catalog."""\nimport json\n\ndef literal(v):return json.dumps(v,ensure_ascii=False,separators=(\',\',\':\'))\ndef ts_type(p):\n    if \'enum\' in p:return \' | \'.join(literal(v) for v in p[\'enum\'])\n    t=p.get(\'type\')\n    if isinstance(t,list):return \' | \'.join(ts_type({**p,\'type\':v}) for v in t)\n    if t==\'array\':return \'(\'+ts_type(p.get(\'items\',{}))+\')[]\'\n    if t==\'object\':\n        fields=p.get(\'properties\')\n        if fields is None:return \'Record<string, unknown>\'\n        base=\'{ \'+ \'; \'.join(literal(k)+(\'\' if k in p.get(\'required\',[]) else \'?\')+\': \'+ts_type(v) for k,v in fields.items())+\' }\'\n        if p.get(\'x-one-of\'):\n            alt=[\'{ \'+\'; \'.join(literal(k)+\': \'+ts_type(fields[k]) for k in group)+\' }\' for group in p[\'x-one-of\']]\n            return base+\' & (\'+\' | \'.join(alt)+\')\'\n        return base\n    return {\'string\':\'string\',\'integer\':\'number\',\'number\':\'number\',\'boolean\':\'boolean\',\'null\':\'null\'}.get(t,\'unknown\')\n\ndef metadata(p,path=\'\',detailed=False):\n    result=[]\n    for k,v in p.items():\n        if k in (\'properties\',\'items\',\'required\',\'type\',\'enum\',\'x-one-of\'):continue\n        if k==\'description\' and not detailed:continue\n        result.append((path+\'.\' if path else \'\')+k+\'=\'+literal(v))\n    if p.get(\'type\')==\'integer\':result.append(path+\':integer\')\n    for k,v in p.get(\'properties\',{}).items():result.extend(metadata(v,(path+\'.\' if path else \'\')+k,detailed))\n    if p.get(\'items\'):result.extend(metadata(p[\'items\'],path+\'[]\',detailed))\n    return result\n\ndef typescript_catalog(specs,detailed=False):\n    lines=[\'interface InstalledTools {\']\n    for name,s in sorted(specs.items()):\n        comment=\'; \'.join([s.get(\'kind\',\'\'),s.get(\'description\',\'\'),*metadata(s[\'parameters\'],detailed=detailed)])\n        if s.get(\'returns\') and detailed:comment+=\'; returns=\'+str(s[\'returns\'])\n        comment=comment.replace(\'*/\',\'* /\').replace(\'\\n\',\' \')\n        lines.append(\'  /** \'+comment+\' */\')\n        lines.append(\'  \'+literal(name)+\'(args: \'+ts_type(s[\'parameters\'])+\'): unknown;\')\n    return \'\\n\'.join(lines+[\'}\'])\n'}
SOURCE_SHA='8830a756fb856bd7bb3db1df8df8134262b5aa750dd5aebc1c04b6867b4f8ab5'
assert hashlib.sha256(json.dumps(PAYLOAD,sort_keys=True).encode()).hexdigest()==SOURCE_SHA
for name,content in PAYLOAD.items():
 p=SOURCE/name;p.parent.mkdir(parents=True,exist_ok=True);p.write_text(content)
inputs=list(Path('/kaggle/input').rglob('probe_manifest.json'));assert len(inputs)==1,'Attach the private evaluation failure inputs'
INPUT=inputs[0].parent;manifest=json.loads(inputs[0].read_text())
assert manifest['source_session_id']==365792172 and manifest['adapter_step']==75 and len(manifest['samples'])==6
for name,wanted in manifest['files'].items():assert hashlib.sha256((INPUT/name).read_bytes()).hexdigest()==wanted,name
(WORK/'input_manifest.json').write_text(json.dumps(manifest,indent=2))
(WORK/'source.json').write_text(json.dumps({'runtime_sha256':SOURCE_SHA,'guided_decoding':False,'experiment':'base_x_lora_dtype_2x2'},indent=2))
print('Selected samples:',manifest['samples'])


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','--quiet','uv==0.9.10'],check=True)
UV=[sys.executable,'-m','uv'];os.environ['UV_NATIVE_TLS']='true'
def runtime():
 path=Path('/tmp/eval_failure_hf');python=str(path/'bin/python')
 if not Path(python).exists():subprocess.run([*UV,'venv','--python','3.12','--seed',str(path)],check=True)
 subprocess.run([*UV,'pip','sync','--index-strategy','unsafe-best-match','--python',python,str(SOURCE/'requirements_train_2gpu.lock')],check=True)
 subprocess.run([*UV,'pip','check','--python',python],check=True)
 purelib=Path(subprocess.check_output([python,'-I','-c',"import sysconfig;print(sysconfig.get_path('purelib'))"],text=True).strip())
 env=os.environ.copy()
 for key in list(env):
  if key in ('PYTHONPATH','PYTHONHOME') or key.startswith(('UNSLOTH_','VLLM_')):env.pop(key,None)
 env.update(PYTHONNOUSERSITE='1',TOKENIZERS_PARALLELISM='false',HF_HOME='/tmp/lifemate_hf_cache',OMP_NUM_THREADS='2',CUDA_VISIBLE_DEVICES='0')
 env['LD_LIBRARY_PATH']=':'.join(str(p) for p in (purelib/'nvidia').glob('*/lib'))+':'+os.environ.get('LD_LIBRARY_PATH','')
 subprocess.run([python,'-c','import torch;assert torch.cuda.device_count()==1'],env=env,check=True)
 return python,env
PYTHON,ENV=runtime()


In [ ]:
# Fresh model processes for each variant, sequential execution, using the same GPU0 for every case.
# Per-case progress is streamed directly to notebook logs and retained in WORK.
for backend in ('nf4_lora_fp32','nf4_lora_fp16','fp16_lora_fp32','fp16_lora_fp16'):
 python,env=PYTHON,ENV
 for variant in ('step75',):
  log=WORK/f'{backend}_{variant}.log'
  command=[python,'-u','-m','umilife.eval_probe','--backend',backend,'--variant',variant,'--input',str(INPUT),'--output',str(WORK)]
  with log.open('w') as stream:
   process=subprocess.Popen(command,cwd=SOURCE,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
   try:
    for line in process.stdout:stream.write(line);stream.flush();print(line,end='',flush=True)
    assert process.wait()==0,f'{backend}/{variant} failed; inspect {log}'
   finally:
    if process.poll() is None:process.terminate();process.wait(timeout=20)


In [ ]:
python,env=PYTHON,ENV
subprocess.run([python,'-u','-m','umilife.probe_summary','--work',str(WORK),'--input',str(INPUT)],cwd=SOURCE,env=env,check=True)
print('Read report.md, report.json, *_settings.json, *.jsonl and *.log in',WORK)
